# Week 67, Day 2: asyncio basics

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
async def greet(name):
    return f"hello {name}"

c = greet("Ada")
print(type(c).__name__)
try:
    c.send(None)
except StopIteration as done:
    print("finished with", repr(done.value))

In [ ]:
import inspect

async def fetch(name):
    return f"{name} page"

print(inspect.iscoroutinefunction(fetch), inspect.iscoroutinefunction(print))
coro = fetch("home")
print(inspect.iscoroutine(coro))
coro.close()                       # tidy up: this coroutine will never be awaited

async def main():
    page = await fetch("home")     # awaiting a coroutine runs it and gives its result
    return page.upper()

c = main()
try:
    c.send(None)
except StopIteration as done:
    print(done.value)

In [ ]:
def concurrency_cost(connections, per_thread_kb=1024, per_task_kb=2):
    "Rough memory for waiting connections: a thread each, or a task each."
    return connections * per_thread_kb // 1024, connections * per_task_kb // 1024

for n in (100, 10_000):
    threads_mb, tasks_mb = concurrency_cost(n)
    print(f"{n} connections: threads about {threads_mb} MB of stack, tasks about {tasks_mb} MB")

In [ ]:
def sequential_time(delays):
    return sum(delays)                 # await one, then the next, then the next

def gathered_time(delays):
    return max(delays, default=0)      # all waiting at the same time

delays = [3, 1, 2]
print(sequential_time(delays), gathered_time(delays))

def finish_order(names, delays):
    "gather returns results in the order given, but they finish in order of delay."
    return [n for n, _ in sorted(zip(names, delays), key=lambda p: p[1])]

print(finish_order(["a", "b", "c"], delays), "finish; gather returns", ["a", "b", "c"])

In [ ]:
class Pause:
    "An awaitable that suspends once, sending a request to whoever drives the coroutine."
    def __init__(self, request):
        self.request = request
    def __await__(self):
        reply = yield self.request         # suspends here; the driver's send() value comes back
        return reply

async def worker():
    first = await Pause(("sleep", 2))
    second = await Pause(("ask", "name?"))
    return f"{first}, {second}"

c = worker()
print(c.send(None))                        # runs to the first yield: the request
print(c.send("woke up"))                   # resumes; the next request
try:
    c.send("Ada")
except StopIteration as done:
    print(done.value)

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Task:
    def __init__(self, coro):
        self.coro, self.done, self.result, self.waiters = coro, False, None, []
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self.seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.ready.extend(task.waiters)
            return
        if kind == "sleep":
            self.seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self.seq, task))
        else:
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

loop = Loop()
log = []

async def fetch(name, delay):
    log.append((loop.now, "start", name))
    await Sleep(delay)
    log.append((loop.now, "done", name))
    return name.upper()

async def main():
    tasks = [loop.create_task(fetch(n, d)) for n, d in [("a", 3), ("b", 1), ("c", 2)]]
    return [await t for t in tasks]          # what gather does

print(loop.run_until_complete(main()), "at time", loop.now)
print(log)

> **Key idea.** async def defines a coroutine function; calling it returns a coroutine object without running it, and await inside another coroutine runs it and gives its result. asyncio.run(main()) runs the top-level coroutine on an event loop, in one thread. The loop runs ready tasks until they await something not ready, then waits for I/O and timers; switching happens only at await (cooperative), so tasks rarely need locks, but a task that blocks or computes freezes all of them. Awaiting coroutines one after another takes the sum of their times; create_task and gather run them concurrently (about the maximum), and gather returns results in the order given. Coroutines are driven with send (StopIteration carries the result) and suspend where an __await__ method yields; that yielded request is what the event loop acts on.

### Exercise: Driving a coroutine by hand

Write `drive(coro)`: run a coroutine to completion by calling `send(None)` repeatedly (whatever it yields is ignored), returning `(result, steps)`, where `steps` is how many times it suspended. Then `Yield`, an awaitable class whose `__await__` suspends exactly once (yielding `None`), so `await Yield()` gives other tasks a turn, like `asyncio.sleep(0)`.

In [ ]:
class Yield:
    ___

def drive(coro):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_drive():\n    \"send and StopIteration\"\n    async def three_steps():\n        await Yield()\n        await Yield()\n        await Yield()\n        return \"done\"\n    assert drive(three_steps()) == (\"done\", 3)\n    async def immediate():\n        return 42\n    assert drive(immediate()) == (42, 0)\n    async def nested():\n        a = await immediate()\n        await Yield()\n        return a + 1\n    assert drive(nested()) == (43, 1)")

### Exercise: One coroutine, simulated time

Write `Sleep(seconds)`, an awaitable that yields `("sleep", seconds)`, and `run_sleeping(coro)`: drive the coroutine, adding up the time it asks to sleep, and return `(result, total_time)`. Requests that aren't sleeps raise `ValueError`. Then `time_of(coro_fn, *args)`: just the total time.

In [ ]:
class Sleep:
    ___

def run_sleeping(coro):
    ___

def time_of(coro_fn, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sleeping():\n    \"a simulated clock\"\n    async def fetch(delay):\n        await Sleep(delay)\n        return delay * 10\n    async def two_in_a_row():\n        a = await fetch(2)\n        b = await fetch(3)\n        return a + b\n    assert run_sleeping(two_in_a_row()) == (50, 5) and time_of(fetch, 4) == 4\n    class Odd:\n        def __await__(self):\n            yield (\"dance\", 1)\n    async def odd():\n        await Odd()\n    try:\n        run_sleeping(odd())\n        assert False\n    except ValueError:\n        pass")

### Exercise: A loop with tasks

Write the tiny event loop: `Sleep(seconds)` yields `("sleep", seconds)`; `Task(coro)` has `done` and `result`, and awaiting an unfinished task yields `("wait", task)` and then returns its result; `Loop` has `now` (starting at 0), `create_task(coro)` (scheduled to run, returns the task), and `run_until_complete(coro)`, which runs everything, returns the coroutine's result, and leaves `now` at the time the last task finished. Ready tasks run in FIFO order; when none are ready, time jumps to the earliest wake-up (ties in sleep order); finished tasks wake their waiters in the order they started waiting.

In [ ]:
import heapq
from collections import deque

class Sleep:
    ___

class Task:
    ___

class Loop:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loop():\n    \"tasks and simulated time\"\n    loop = Loop()\n    log = []\n    async def fetch(name, delay):\n        log.append((loop.now, \"start\", name))\n        await Sleep(delay)\n        log.append((loop.now, \"done\", name))\n        return name * 2\n    async def main():\n        tasks = [loop.create_task(fetch(n, d)) for n, d in [(\"a\", 3), (\"b\", 1), (\"c\", 2)]]\n        return [await t for t in tasks]\n    assert loop.run_until_complete(main()) == [\"aa\", \"bb\", \"cc\"] and loop.now == 3\n    assert log == [(0, \"start\", \"a\"), (0, \"start\", \"b\"), (0, \"start\", \"c\"),\n                   (1, \"done\", \"b\"), (2, \"done\", \"c\"), (3, \"done\", \"a\")]\n    loop2 = Loop()\n    async def one_by_one():\n        return [await fetch(\"x\", 2), await fetch(\"y\", 2)]\n    assert loop2.run_until_complete(one_by_one()) == [\"xx\", \"yy\"] and loop2.now == 4\n    loop3 = Loop()\n    async def quick():\n        return 7\n    async def waits_for_done():\n        t = loop3.create_task(quick())\n        await Sleep(5)\n        return await t\n    assert loop3.run_until_complete(waits_for_done()) == 7 and loop3.now == 5")

### Exercise: gather

Using the loop below (the one from the previous exercise, given here), write a coroutine `gather(loop, *coros)` that runs the coroutines concurrently as tasks and returns their results **in the order given**, and a coroutine `first_done(loop, *coros)` that returns the result of whichever finishes **first** in simulated time (ties: the earliest given), without waiting for the others to finish (they keep running in the background).

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Task:
    def __init__(self, coro):
        self.coro, self.done, self.result, self.waiters = coro, False, None, []
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.ready.extend(task.waiters)
            task.waiters = []
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

async def gather(loop, *coros):
    ___

async def first_done(loop, *coros):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gather():\n    \"concurrent coroutines\"\n    loop = Loop()\n    async def job(name, delay):\n        await Sleep(delay)\n        return name\n    async def main():\n        results = await gather(loop, job(\"slow\", 5), job(\"fast\", 1), job(\"mid\", 3))\n        return results, loop.now\n    assert loop.run_until_complete(main()) == ([\"slow\", \"fast\", \"mid\"], 5)\n    loop2 = Loop()\n    async def race():\n        winner = await first_done(loop2, job(\"slow\", 5), job(\"fast\", 1), job(\"mid\", 3))\n        return winner, loop2.now\n    assert loop2.run_until_complete(race()) == (\"fast\", 1) and loop2.now == 5\n    loop3 = Loop()\n    async def empty():\n        return await gather(loop3)\n    assert loop3.run_until_complete(empty()) == []")

### Exercise: Async generators

`async def` functions that `yield` are **async generators**, consumed with `async for`. Write an async generator `countdown(n)` yielding `n, n - 1, ..., 1` and awaiting `Pause()` (a provided awaitable that suspends once) before each value, and a coroutine `collect(agen)` that gathers everything an async generator yields into a list with `async for`. Then `drive(coro)` returning `(result, suspensions)` as in the first exercise, to run `collect` here.

In [ ]:
class Pause:
    def __await__(self):
        yield

async def countdown(n):
    ___

async def collect(agen):
    ___

def drive(coro):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_agen():\n    \"async for\"\n    import inspect\n    assert inspect.isasyncgenfunction(countdown)\n    assert drive(collect(countdown(3))) == ([3, 2, 1], 3)\n    assert drive(collect(countdown(0))) == ([], 0)\n    async def doubled():\n        out = []\n        async for x in countdown(2):\n            out.append(x * 2)\n        return out\n    assert drive(doubled())[0] == [4, 2]")

### Bug hunt: A coroutine that never ran

`main` calls `fetch(...)` but doesn't `await` it, so it gets a coroutine object instead of the page, and this prints `coroutine`. Await the call so it prints `HOME PAGE`.

In [ ]:
async def fetch(name):
    return f"{name} page"

async def main():
    page = fetch("home")
    return page.upper() if isinstance(page, str) else type(page).__name__

c = main()
try:
    c.send(None)
except StopIteration as done:
    print(done.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"await it\"\n    expect_output(\"HOME PAGE\")")

### Bug hunt: Concurrent in name only

The three downloads are awaited one after another, so their waits add up and this prints `6`. Start them all as tasks first and then await them, so they wait at the same time and this prints `3`.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, s):
        self.s = s
    def __await__(self):
        yield ("sleep", self.s)

class Task:
    def __init__(self, coro):
        self.coro, self.done, self.result, self.waiters = coro, False, None, []
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self.seq = 0, deque(), [], 0
    def create_task(self, coro):
        t = Task(coro)
        self.ready.append(t)
        return t
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, t = heapq.heappop(self.sleepers)
                self.ready.append(t)
                continue
            t = self.ready.popleft()
            try:
                kind, arg = t.coro.send(None)
            except StopIteration as stop:
                t.done, t.result = True, stop.value
                self.ready.extend(t.waiters)
                continue
            if kind == "sleep":
                self.seq += 1
                heapq.heappush(self.sleepers, (self.now + arg, self.seq, t))
            else:
                arg.waiters.append(t)
        return main.result

loop = Loop()

async def download(seconds):
    await Sleep(seconds)

async def main():
    for s in (3, 1, 2):
        await download(s)

loop.run_until_complete(main())
print(loop.now)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"create tasks first\"\n    expect_output(\"3\")")

### Bug hunt: A result lost in StopIteration

The driver catches `StopIteration` but throws away the coroutine's return value, which travels **inside** the exception, so this prints `None`. Take the result from the exception's `value` so it prints `42`.

In [ ]:
async def answer():
    return 42

def drive(coro):
    result = None
    try:
        while True:
            coro.send(None)
    except StopIteration:
        pass
    return result

print(drive(answer()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"StopIteration.value\"\n    expect_output(\"42\")")

### Puzzle: Running coroutines concurrently

Arrange the lines of a `main` coroutine that runs three fetches at the same time and prints the pages in the order requested (as you'd write it with real `asyncio`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what calling an async def function returns: "its result" or "a coroutine"
q = ___   # three awaits in a row of 2, 3 and 1 seconds take how long in total?
r = ___   # the same three run with gather take how long?
s = ___   # True or False: a task can be paused by asyncio between two lines that have no await
t = ___   # the exception whose value carries a finished coroutine's return value, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"a coroutine\" and q == 6 and r == 3, \"p, q and r\"\n    assert s is False and t == \"StopIteration\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: timeouts and cancellation

Extend the tiny loop so tasks can be **cancelled**, which is how `asyncio.wait_for` implements timeouts. Write `Sleep`, `Task`, `Loop` and these, using a `Cancelled` exception class:

- `task.cancel()`: if the task hasn't finished, it's marked to be cancelled; the next time the loop would run it (it's woken at once if it was sleeping or waiting), the loop calls `task.coro.throw(Cancelled())` instead of `send(None)`. If the coroutine lets the exception escape, the task finishes as **cancelled** (`task.cancelled` is True); awaiting a cancelled task raises `Cancelled`. A coroutine may catch `Cancelled` to clean up (in a `finally`).
- An exception escaping any other task is stored in `task.error`, and awaiting that task re-raises it.
- `wait_for(loop, coro, timeout)`: a coroutine that runs `coro` as a task and returns its result, but if it hasn't finished after `timeout` units of simulated time, cancels it and raises `TimeoutError`.
- `run_until_complete(coro)` returns the main task's result, or raises its error.

In [ ]:
import heapq
from collections import deque

class Cancelled(Exception):
    pass

class Sleep:
    ___

class Task:
    ___

class Loop:
    ___

async def wait_for(loop, coro, timeout):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timeouts():\n    \"wait_for\"\n    loop = Loop()\n    log = []\n    async def fetch(delay):\n        try:\n            await Sleep(delay)\n            return \"page\"\n        finally:\n            log.append((\"cleanup\", loop.now))\n    async def main():\n        quick = await wait_for(loop, fetch(2), timeout=5)\n        try:\n            await wait_for(loop, fetch(10), timeout=3)\n        except TimeoutError:\n            return quick, \"timed out\", loop.now\n    assert loop.run_until_complete(main()) == (\"page\", \"timed out\", 5)\n    assert log == [(\"cleanup\", 2), (\"cleanup\", 5)]\n\ndef test_cancel_and_errors():\n    \"cancellation and exceptions\"\n    loop = Loop()\n    async def forever():\n        while True:\n            await Sleep(1)\n    async def broken():\n        await Sleep(1)\n        raise KeyError(\"missing\")\n    async def main():\n        t = loop.create_task(forever())\n        await Sleep(3)\n        t.cancel()\n        try:\n            await t\n        except Cancelled:\n            pass\n        b = loop.create_task(broken())\n        try:\n            await b\n        except KeyError:\n            return t.cancelled, isinstance(b.error, KeyError), loop.now\n    assert loop.run_until_complete(main()) == (True, True, 4)\n    loop2 = Loop()\n    async def fails():\n        raise ValueError(\"bad\")\n    try:\n        loop2.run_until_complete(fails())\n        assert False\n    except ValueError:\n        pass")

## Warm-up

Calling an `async def` function doesn't run it: it returns a **coroutine** object. The code runs only when something drives the coroutine (here, `send(None)`), and when it finishes, its return value travels inside `StopIteration`, exactly as for a generator. An event loop is just a program that does this driving for many coroutines at once.

## Coroutines and await

- **`async def f(...)`** defines a **coroutine function**. Calling it returns a coroutine object, without running any of the body.
- **`await x`** inside a coroutine pauses it until the **awaitable** `x` has a result, then gives back that result. You can only `await` inside `async def`.
- **Awaitables** are coroutines, `asyncio` **tasks** and **futures**, and any object with an `__await__` method.
- **`asyncio.run(main())`** creates an event loop, runs the coroutine `main()` to completion, and closes the loop. It's normally called once, at the top of the program.
- `await asyncio.sleep(1)` pauses for a second **without blocking** the thread: other tasks run meanwhile. (`time.sleep(1)` would block everything: Thursday.)

On your own computer:

```
import asyncio

async def fetch(name, delay):
    print("start", name)
    await asyncio.sleep(delay)          # stands for waiting on the network
    print("done", name)
    return f"{name} page"

async def main():
    page = await fetch("home", 1)       # waits for this one; nothing else is running
    print(page)

asyncio.run(main())
```

## The event loop

The **event loop** is the scheduler at the heart of `asyncio`. It keeps a set of **tasks** (coroutines it's responsible for), and repeatedly:

1. runs each ready task until it reaches an `await` on something that isn't ready yet, at which point the task gives control back to the loop;
2. asks the operating system which network sockets and files have data (with one efficient call, such as `select` or `epoll`), and which timers have expired;
3. marks the tasks waiting for those as ready again.

Two consequences follow:

- **Switching is cooperative**: a task is only ever paused at an `await`. Between two `await`s, a task runs without interruption, so most code needs **no locks** (unlike threads). The price: a task that computes for a long time, or calls a blocking function, freezes **every** task (Thursday).
- **One thread can serve thousands of connections**, because a waiting task costs only a small object, not a thread with its own stack. That's why web servers and chat services use `asyncio`.

`asyncio` helps with **I/O-bound** work only: CPU-bound work still needs processes (yesterday).

## Tasks and gather

`await fetch(...)` runs one coroutine to the end before the next line, so three awaits in a row take the **sum** of their times. To run coroutines **concurrently**:

- **`asyncio.create_task(coro)`** wraps a coroutine in a **task** and schedules it on the loop at once; it starts running the next time the current task awaits. Awaiting the task later gives its result. Keep a reference to every task you create (Thursday explains why).
- **`await asyncio.gather(*coros)`** runs several awaitables concurrently and returns their results **in the order given**, taking about as long as the slowest.
- **`asyncio.TaskGroup`** (3.11+) is the structured alternative: `async with asyncio.TaskGroup() as tg: tg.create_task(...)` waits for all its tasks at the end of the block and cancels the rest if one fails (Wednesday).

On your own computer:

```
async def main():
    start = time.perf_counter()
    pages = await asyncio.gather(fetch("a", 3), fetch("b", 1), fetch("c", 2))
    print(pages, round(time.perf_counter() - start, 1), "seconds")   # about 3, not 6

asyncio.run(main())
```

## Inside a coroutine

Coroutines are built on the same machinery as generators (Week 59):

- `coro.send(value)` runs the coroutine until it next **suspends**, and returns whatever it yielded to the loop; the value sent becomes the result of the expression where it was suspended. `send(None)` starts it.
- When the coroutine returns, `send` raises **`StopIteration`** carrying the return value.
- `coro.throw(exc)` raises an exception inside it at the point where it's suspended (that's how cancellation works: Thursday), and `coro.close()` stops it.

Where does a coroutine actually suspend? `await` passes control down a chain of awaitables until it reaches an object whose **`__await__`** method **yields**. That yielded value travels all the way up to whoever called `send`: the event loop. In `asyncio`, it's a future that the loop should wait for; in our own loop, it can be any **request** we choose, such as `("sleep", 2)`.

## A tiny event loop

With awaitables that yield requests, an event loop takes about thirty lines. Ours uses **simulated time**, so it runs instantly and the same way every time:

- a **`Sleep(t)`** awaitable yields `("sleep", t)`;
- a **`Task`** wraps a coroutine; awaiting a task that isn't finished yields `("wait", task)`;
- the **loop** keeps a queue of ready tasks and a heap of sleeping ones, runs each ready task until its next request with `send(None)`, and when a task finishes, wakes the tasks waiting for it. When nothing is ready, time jumps to the next wake-up.

`asyncio`'s real loop has the same structure, with sockets and timers in place of simulated sleeps.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run the `gather` example with real `asyncio` and check that three downloads of 3, 1 and 2 seconds take about 3 seconds. Then add `print` calls to see the order in which they start and finish. Tomorrow: the patterns that real async programs use, from limiting concurrency to queues and timeouts.